# Notebook 00 — Build the Kerala LSGD Feature Store

Builds one row per panchayat/municipality/corporation (1,034 LSGD units) with terrain,
rainfall, water-proximity, and land-cover features, then attaches district-level 2018
flood/landslide labels.

**Files to upload when prompted:**
- `kerala_lsgd_boundaries.geojson` (1,034 LSGD polygons — columns: lsgd_id, lsgd_name,
  lsgd_type, district, block_name, lsgi_code)
- `district_labels_2018.csv` (compiled from KSDMA reports)

**Output:** `lsgd_feature_store.csv`

In [1]:
!pip install -q geemap geopandas earthengine-api

import ee, geemap, geopandas as gpd, pandas as pd, numpy as np
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/DIP_Kerala'

ee.Authenticate()
ee.Initialize(project='dip-kerala-502817')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.1 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
lsgd = gpd.read_file(f'{BASE}/00_boundaries/kerala_lsgd_boundaries.geojson')
print("Total local bodies loaded:", len(lsgd))
print("Columns found:", lsgd.columns.tolist())

# Handles both the raw OSM export (local_auth/name/District) and the pre-cleaned
# version (lsgd_type/lsgd_name/district) -- whichever one actually got uploaded
rename_map = {
    'local_auth': 'lsgd_type',
    'name': 'lsgd_name',
    'name_ml': 'lsgd_name_ml',
    'District': 'district',
    'BlockName': 'block_name',
    'LSGI_Code': 'lsgi_code',
}
lsgd = lsgd.rename(columns={k: v for k, v in rename_map.items() if k in lsgd.columns})

if 'lsgd_id' not in lsgd.columns:
    lsgd['lsgd_id'] = range(1, len(lsgd) + 1)

required = ['lsgd_id', 'lsgd_name', 'lsgd_type', 'district']
missing = [c for c in required if c not in lsgd.columns]
if missing:
    raise ValueError(f"Missing columns after rename: {missing}. Actual columns: {lsgd.columns.tolist()}")

print(lsgd['lsgd_type'].value_counts())
lsgd.head()

Total local bodies loaded: 1034
Columns found: ['admin_leve', 'local_auth', 'name', 'name_ml', 'wikidata', 'LSGI_Code', 'District', 'Block_QID', 'BlockName', 'DP_QID', 'DP_Name', 'geometry']
lsgd_type
gram_panchayat           941
municipality              87
municipal_corporation      6
Name: count, dtype: int64


,admin_leve,lsgd_type,lsgd_name,lsgd_name_ml,wikidata,lsgi_code,district,Block_QID,block_name,DP_QID,DP_Name,geometry,lsgd_id
0,8,gram_panchayat,Vorkady,വോര്‍ക്കാടി,Q16134001,G14009,Kasaragod,Q16134202,Manjeswaram Block Panchayat,Q61368450,Kasaragod District Panchayat,"MULTIPOLYGON (((74.90055 12.76391, 74.90136 12...",1
1,8,gram_panchayat,Manjeswaram,മഞ്ചേശ്വരം,Q16134195,G14008,Kasaragod,Q16134202,Manjeswaram Block Panchayat,Q61368450,Kasaragod District Panchayat,"MULTIPOLYGON (((74.88595 12.70858, 74.88558 12...",2
2,8,gram_panchayat,Paivalike,പൈവെളിഗെ,Q13113487,G14012,Kasaragod,Q16134202,Manjeswaram Block Panchayat,Q61368450,Kasaragod District Panchayat,"MULTIPOLYGON (((74.96049 12.67596, 74.96117 12...",3
3,8,gram_panchayat,Meenja,മീഞ്ച,Q13114089,G14010,Kasaragod,Q16134202,Manjeswaram Block Panchayat,Q61368450,Kasaragod District Panchayat,"MULTIPOLYGON (((74.96049 12.67596, 74.95909 12...",4
4,8,gram_panchayat,Mangalpaddy,മംഗൽപ്പാടി,Q16134041,G14011,Kasaragod,Q16134202,Manjeswaram Block Panchayat,Q61368450,Kasaragod District Panchayat,"MULTIPOLYGON (((74.88595 12.70858, 74.88672 12...",5


In [3]:
lsgd_wgs84 = lsgd.to_crs(epsg=4326)
lsgd_wgs84_simplified = lsgd_wgs84.copy()
lsgd_wgs84_simplified.geometry = lsgd_wgs84_simplified.geometry.simplify(tolerance=0.0001, preserve_topology=True)  # reduces payload size for GEE upload
lsgd_fc = geemap.geopandas_to_ee(lsgd_wgs84_simplified)
print("Feature count in GEE:", lsgd_fc.size().getInfo())

Feature count in GEE: 1034


## Terrain features — elevation & slope (SRTM DEM)

In [4]:
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem)
terrain_stack = dem.rename('elevation').addBands(slope.rename('slope'))

terrain_stats = terrain_stack.reduceRegions(collection=lsgd_fc, reducer=ee.Reducer.mean(), scale=30)
terrain_df = geemap.ee_to_df(terrain_stats)[['lsgd_id', 'elevation', 'slope']]
terrain_df.head()

,lsgd_id,elevation,slope
0,1,59.011230,7.386920
1,2,21.380619,4.270908
2,3,88.465243,8.798889
3,4,52.726992,6.329972
4,5,23.913829,4.745512


## Rainfall features — Open-Meteo (7-day antecedent rainfall)

**Why this changed from CHIRPS:** the flood/landslide models will eventually need to accept
*live* rainfall at inference time (Notebook 04), fetched from a weather API. If training used
a 2.5-month season TOTAL (400-1500mm scale) but live queries feed in a single day's rainfall
(0-100mm scale), the model would be extrapolating wildly outside anything it learned from.

The fix: train on the same **7-day antecedent rainfall** window (ending at the 2018 flood
peak, 13-19 Aug) that we'll also compute live for current queries -- same time-window, same
data source (Open-Meteo), so training and inference are consistent. No new dataset needed;
this just changes which API supplies the rainfall number and over what window.

In [5]:
import requests
import time

# 7-day window ending at the 2018 Kerala flood peak (matches what we'll fetch live in Notebook 04)
TRAIN_START_DATE = '2018-08-13'
TRAIN_END_DATE = '2018-08-19'

# Open-Meteo needs a lat/lon point per call, not a polygon -- use each LSGD's centroid.
# Free tier: no API key, no card, ~10,000 calls/day non-commercial -- 1,034 calls fits easily.
lsgd_wgs84_pts = lsgd.to_crs(epsg=4326).copy()
lsgd_wgs84_pts['centroid'] = lsgd_wgs84_pts.geometry.centroid
lsgd_wgs84_pts['lat'] = lsgd_wgs84_pts['centroid'].y
lsgd_wgs84_pts['lon'] = lsgd_wgs84_pts['centroid'].x

def fetch_7day_rainfall(lat, lon, start_date, end_date, retries=3):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "daily": "precipitation_sum", "timezone": "auto",
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=15)
            r.raise_for_status()
            data = r.json()
            values = data.get('daily', {}).get('precipitation_sum', [])
            values = [v for v in values if v is not None]
            return sum(values) if values else 0.0
        except Exception as e:
            if attempt == retries - 1:
                print(f"Failed for ({lat:.3f},{lon:.3f}) after {retries} tries: {e}")
                return None
            time.sleep(1)

rainfall_records = []
for idx, row in lsgd_wgs84_pts.iterrows():
    rain_mm = fetch_7day_rainfall(row['lat'], row['lon'], TRAIN_START_DATE, TRAIN_END_DATE)
    rainfall_records.append({'lsgd_id': row['lsgd_id'], 'rainfall_7day_mm': rain_mm})
    if idx % 100 == 0:
        print(f"Progress: {idx}/{len(lsgd_wgs84_pts)}")

rainfall_df = pd.DataFrame(rainfall_records)
rainfall_df['rainfall_7day_mm'] = rainfall_df['rainfall_7day_mm'].fillna(rainfall_df['rainfall_7day_mm'].mean())
rainfall_df['event_year'] = 2018
print(f"\nFetched rainfall for {len(rainfall_df)} units. Failed/filled: {rainfall_df['rainfall_7day_mm'].isna().sum()}")
rainfall_df.head()

/tmp/ipykernel_492/152387459.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lsgd_wgs84_pts['centroid'] = lsgd_wgs84_pts.geometry.centroid


Progress: 0/1034
Progress: 100/1034
Progress: 200/1034
Progress: 300/1034
Progress: 400/1034
Progress: 500/1034
Progress: 600/1034
Progress: 700/1034
Progress: 800/1034
Progress: 900/1034
Progress: 1000/1034

Fetched rainfall for 1034 units. Failed/filled: 0


,lsgd_id,rainfall_7day_mm,event_year
0,1,285.2,2018
1,2,230.5,2018
2,3,285.9,2018
3,4,252.0,2018
4,5,252.0,2018


## Water proximity — distance to permanent surface water (JRC)

In [6]:
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
water_mask = gsw.gt(50)
distance_to_water = water_mask.fastDistanceTransform().sqrt().multiply(ee.Image.pixelArea().sqrt()).rename('dist_to_water_m')

water_stats = distance_to_water.reduceRegions(collection=lsgd_fc, reducer=ee.Reducer.mean(), scale=100)
water_df = geemap.ee_to_df(water_stats)[['lsgd_id', 'mean']].rename(columns={'mean': 'dist_to_water_m'})  # same GEE naming behavior as rainfall above
water_df.head()

,lsgd_id,dist_to_water_m
0,1,4374.374389
1,2,1516.365486
2,3,7627.278746
3,4,3464.459376
4,5,1553.752487


## Land cover — vegetation / built-up % (ESA WorldCover)

In [7]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
veg_mask = worldcover.eq(10).Or(worldcover.eq(40)).rename('vegetation')
builtup_mask = worldcover.eq(50).rename('builtup')
landcover_stack = veg_mask.addBands(builtup_mask)

landcover_stats = landcover_stack.reduceRegions(collection=lsgd_fc, reducer=ee.Reducer.mean(), scale=10)
landcover_df = geemap.ee_to_df(landcover_stats)[['lsgd_id', 'vegetation', 'builtup']]
landcover_df.head()

,lsgd_id,vegetation,builtup
0,1,0.937367,0.012008
1,2,0.800916,0.080076
2,3,0.877275,0.013239
3,4,0.850222,0.018589
4,5,0.856307,0.049864


## Merge terrain + rainfall + water + land cover

In [8]:
feature_store = lsgd[['lsgd_id', 'lsgd_name', 'lsgd_type', 'district']].merge(terrain_df, on='lsgd_id') \
    .merge(rainfall_df, on='lsgd_id') \
    .merge(water_df, on='lsgd_id') \
    .merge(landcover_df, on='lsgd_id')

print(feature_store.shape)
feature_store.head()

(1034, 11)


,lsgd_id,lsgd_name,lsgd_type,district,elevation,slope,rainfall_7day_mm,event_year,dist_to_water_m,vegetation,builtup
0,1,Vorkady,gram_panchayat,Kasaragod,59.011230,7.386920,285.2,2018,4374.374389,0.937367,0.012008
1,2,Manjeswaram,gram_panchayat,Kasaragod,21.380619,4.270908,230.5,2018,1516.365486,0.800916,0.080076
2,3,Paivalike,gram_panchayat,Kasaragod,88.465243,8.798889,285.9,2018,7627.278746,0.877275,0.013239
3,4,Meenja,gram_panchayat,Kasaragod,52.726992,6.329972,252.0,2018,3464.459376,0.850222,0.018589
4,5,Mangalpaddy,gram_panchayat,Kasaragod,23.913829,4.745512,252.0,2018,1553.752487,0.856307,0.049864


## Attach district-level labels

Upload `district_labels_2018.csv` — every LSGD unit inherits its district's flood/landslide
outcome for 2018. This is a documented scoping choice — true panchayat-level ground truth
isn't available in any official secondary source.

In [9]:
labels_df = pd.read_csv(f'{BASE}/01_labels/district_labels_2018.csv')

feature_store_labeled = feature_store.merge(labels_df, on='district', how='left')

print(feature_store_labeled['flood_occurred'].value_counts())
print(feature_store_labeled['landslide_occurred'].value_counts())
feature_store_labeled.head()

flood_occurred
1.0    581
0.0    412
Name: count, dtype: int64
landslide_occurred
1.0    532
0.0    461
Name: count, dtype: int64


,lsgd_id,lsgd_name,lsgd_type,district,elevation,slope,rainfall_7day_mm,event_year_x,dist_to_water_m,vegetation,...,fully_damaged_houses,severely_damaged_houses,roads_damaged_km,bridges_damaged,severity_score,flood_risk_level,landslide_risk_level,flood_occurred,landslide_occurred,event_year_y
0,1,Vorkady,gram_panchayat,Kasaragod,59.011230,7.386920,285.2,2018,4374.374389,0.937367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Manjeswaram,gram_panchayat,Kasaragod,21.380619,4.270908,230.5,2018,1516.365486,0.800916,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Paivalike,gram_panchayat,Kasaragod,88.465243,8.798889,285.9,2018,7627.278746,0.877275,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Meenja,gram_panchayat,Kasaragod,52.726992,6.329972,252.0,2018,3464.459376,0.850222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Mangalpaddy,gram_panchayat,Kasaragod,23.913829,4.745512,252.0,2018,1553.752487,0.856307,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
import os
os.makedirs(f'{BASE}/03_processed', exist_ok=True)
feature_store_labeled.to_csv(f'{BASE}/03_processed/lsgd_feature_store.csv', index=False)
print('Saved to', f'{BASE}/03_processed/lsgd_feature_store.csv')
print('Feeds directly into 01_Flood_Prediction_Model.ipynb and 02_Landslide_Prediction_Model.ipynb')

Saved to /content/drive/MyDrive/DIP_Kerala/03_processed/lsgd_feature_store.csv
Feeds directly into 01_Flood_Prediction_Model.ipynb and 02_Landslide_Prediction_Model.ipynb


## Known limitation — state this in your report

All 1,034 LSGD units within the same district share the same `flood_occurred` /
`landslide_occurred` label, since no official source publishes true panchayat-level
ground truth for 2018. Terrain/rainfall features still vary per LSGD (real signal),
but the label is a district-level proxy. This is an explicit, documented scoping
decision — not an oversight.

## Feature Store Summary

**Note on 'accuracy' for this notebook:** Notebook 00 doesn't train a model -- it builds
the feature table that Notebooks 01, 02, and 04 train on. There's no accuracy metric to
report here. Instead, this cell reports data completeness and label balance, which is the
equivalent sanity check for a data-pipeline notebook -- catching missing values or broken
GEE/API fetches before they silently corrupt downstream model training.

In [11]:
print("="*50)
print("FEATURE STORE SUMMARY")
print("="*50)
print(f"Total LSGD units: {len(feature_store_labeled)}")
print(f"\nMissing values per column:")
print(feature_store_labeled.isnull().sum())
print(f"\nUnits per LSGD type:")
print(feature_store_labeled['lsgd_type'].value_counts())
print(f"\nflood_occurred distribution:")
print(feature_store_labeled['flood_occurred'].value_counts())
print(f"\nlandslide_occurred distribution:")
print(feature_store_labeled['landslide_occurred'].value_counts())
print(f"\nrainfall_7day_mm range: {feature_store_labeled['rainfall_7day_mm'].min():.1f} - {feature_store_labeled['rainfall_7day_mm'].max():.1f} mm")
print(f"elevation range: {feature_store_labeled['elevation'].min():.1f} - {feature_store_labeled['elevation'].max():.1f} m")
print("="*50)

FEATURE STORE SUMMARY
Total LSGD units: 1034

Missing values per column:
lsgd_id                     0
lsgd_name                   0
lsgd_type                   0
district                    0
elevation                   0
slope                       0
rainfall_7day_mm            0
event_year_x                0
dist_to_water_m             0
vegetation                  0
builtup                     0
fatalities                 41
no_of_landslides           41
agri_crop_loss_ha          41
fully_damaged_houses       41
severely_damaged_houses    41
roads_damaged_km           41
bridges_damaged            41
severity_score             41
flood_risk_level           41
landslide_risk_level       41
flood_occurred             41
landslide_occurred         41
event_year_y               41
dtype: int64

Units per LSGD type:
lsgd_type
gram_panchayat           941
municipality              87
municipal_corporation      6
Name: count, dtype: int64

flood_occurred distribution:
flood_occurred
1.0 